# 📈 Análisis de Portafolio de Activos — Mercado LATAM

**Objetivo:** Aplicar técnicas cuantitativas de finanzas para analizar riesgo/retorno de activos financieros representativos del mercado latinoamericano, con foco en el contexto boliviano.

**Técnicas aplicadas:**
- Retornos logarítmicos y estadísticos descriptivos
- Matriz de correlaciones
- Sharpe Ratio por activo
- Value at Risk (VaR) paramétrico al 95%
- Simulación Monte Carlo de portafolios
- Frontera eficiente de Markowitz

**Activos analizados:** ETFs representativos de LATAM (ILF), mercados emergentes (EEM), commodities (GLD, SLV) y renta fija (IEF) — relevantes para cualquier análisis de diversificación desde Bolivia.

In [ ]:
# ─── DEPENDENCIAS ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yfinance as yf
from scipy import stats
from scipy.stats import norm, jarque_bera
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficas
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✅ Librerías cargadas correctamente')

## 1. Carga de Datos

In [ ]:
# ─── CONFIGURACIÓN ───────────────────────────────────────────────────────────
TICKERS = {
    'ILF':  'iShares LATAM 40 ETF',
    'EEM':  'iShares Mercados Emergentes',
    'GLD':  'SPDR Gold ETF',
    'SLV':  'iShares Silver ETF',
    'IEF':  'iShares Bonos Tesoro 7-10y',
}

FECHA_INICIO = '2020-01-01'
FECHA_FIN    = '2024-12-31'
TASA_LIBRE_RIESGO = 0.045  # Aproximado: rendimiento bonos tesoro USA 2024

# ─── DESCARGA ────────────────────────────────────────────────────────────────
print(f'Descargando datos {FECHA_INICIO} → {FECHA_FIN}...')
raw = yf.download(list(TICKERS.keys()), start=FECHA_INICIO, end=FECHA_FIN, auto_adjust=True)
precios = raw['Close'].dropna()

print(f'\n✅ {len(precios)} días de trading descargados')
print(f'   Activos: {list(precios.columns)}')
precios.tail(3)

## 2. Retornos y Estadísticos Descriptivos

In [ ]:
# ─── RETORNOS LOGARÍTMICOS ───────────────────────────────────────────────────
# Usamos log-retornos porque son aditivos en el tiempo y se comportan
# mejor estadísticamente que retornos simples para periodos largos
retornos = np.log(precios / precios.shift(1)).dropna()

# ─── ESTADÍSTICOS ANUALIZADOS ────────────────────────────────────────────────
dias_trading = 252

stats_df = pd.DataFrame({
    'Activo':           [TICKERS[t] for t in retornos.columns],
    'Retorno Anual (%)': (retornos.mean() * dias_trading * 100).round(2),
    'Volatilidad (%)':   (retornos.std() * np.sqrt(dias_trading) * 100).round(2),
    'Asimetría':         retornos.skew().round(3),
    'Curtosis':          retornos.kurtosis().round(3),
    'Max Drawdown (%)':  [round(((precios[t] / precios[t].cummax()) - 1).min() * 100, 2)
                          for t in precios.columns],
}).set_index('Activo')

# Sharpe Ratio por activo
stats_df['Sharpe Ratio'] = (
    (stats_df['Retorno Anual (%)'] / 100 - TASA_LIBRE_RIESGO) /
    (stats_df['Volatilidad (%)'] / 100)
).round(3)

print('📊 Estadísticos descriptivos anualizados:\n')
stats_df

In [ ]:
# ─── GRÁFICA: EVOLUCIÓN DE PRECIOS NORMALIZADOS ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Precios normalizados a 100
ax = axes[0]
(precios / precios.iloc[0] * 100).plot(ax=ax)
ax.set_title('Evolución de Precios (Base 100)', fontweight='bold')
ax.set_ylabel('Valor (base 100)')
ax.set_xlabel('')
ax.legend([TICKERS[t] for t in precios.columns], fontsize=9)

# Distribución de retornos diarios
ax2 = axes[1]
for ticker in retornos.columns:
    retornos[ticker].hist(ax=ax2, bins=80, alpha=0.5, label=ticker, density=True)
ax2.set_title('Distribución de Retornos Diarios', fontweight='bold')
ax2.set_xlabel('Retorno diario')
ax2.set_ylabel('Densidad')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('img/01_precios_y_distribuciones.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfica guardada en img/')

## 3. Correlaciones y Diversificación

In [ ]:
# ─── MATRIZ DE CORRELACIÓN ───────────────────────────────────────────────────
corr = retornos.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # Ocultar triángulo superior
sns.heatmap(
    corr,
    annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=-1, vmax=1, center=0,
    square=True, ax=ax,
    xticklabels=corr.columns,
    yticklabels=corr.columns
)
ax.set_title('Matriz de Correlación de Retornos Diarios', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('img/02_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Interpretación:')
print('   Correlación alta (>0.7)  → poca diversificación entre esos activos')
print('   Correlación baja (<0.3)  → buena diversificación potencial')
print('   Correlación negativa     → cobertura natural (hedge)')

## 4. Value at Risk (VaR)

In [ ]:
# ─── VALUE AT RISK PARAMÉTRICO AL 95% ────────────────────────────────────────
# VaR(95%) = pérdida máxima esperada en un día con 95% de confianza
# Asumimos distribución normal (VaR paramétrico)

CONFIANZA = 0.95
INVERSION = 100_000  # USD - inversión hipotética

var_results = []
for ticker in retornos.columns:
    mu    = retornos[ticker].mean()
    sigma = retornos[ticker].std()

    # VaR paramétrico
    var_pct    = norm.ppf(1 - CONFIANZA, mu, sigma)
    var_usd    = var_pct * INVERSION

    # VaR histórico (no paramétrico - como comparación)
    var_hist   = retornos[ticker].quantile(1 - CONFIANZA)

    # Test Jarque-Bera: ¿son normales los retornos?
    jb_stat, jb_p = jarque_bera(retornos[ticker])

    var_results.append({
        'Ticker':         ticker,
        'VaR 95% (%)':    round(var_pct * 100, 3),
        'VaR 95% (USD)':  round(var_usd, 0),
        'VaR Histórico (%)': round(var_hist * 100, 3),
        'Normal? (p>0.05)': 'No' if jb_p < 0.05 else 'Sí',
    })

var_df = pd.DataFrame(var_results).set_index('Ticker')
print(f'📉 Value at Risk — Inversión hipotética: ${INVERSION:,} USD\n')
print(var_df.to_string())
print(f'\n⚠️  Nota: Si "Normal?" es No → el VaR paramétrico subestima el riesgo real.')
print(f'   En ese caso, el VaR Histórico es más conservador y recomendable.')

## 5. Simulación Monte Carlo — Frontera Eficiente

In [ ]:
# ─── MONTE CARLO: 20,000 PORTAFOLIOS ALEATORIOS ──────────────────────────────
# Generamos portafolios aleatorios para mapear el espacio riesgo/retorno
# y encontrar la frontera eficiente de Markowitz

N_SIMULACIONES = 20_000
n_activos      = len(retornos.columns)

media_anual = retornos.mean() * dias_trading
cov_anual   = retornos.cov()  * dias_trading

np.random.seed(42)
resultados = np.zeros((3, N_SIMULACIONES))  # [retorno, volatilidad, sharpe]
pesos_mc   = np.zeros((n_activos, N_SIMULACIONES))

for i in range(N_SIMULACIONES):
    w = np.random.random(n_activos)
    w /= w.sum()  # Normalizar a 100%

    ret  = np.dot(w, media_anual)
    vol  = np.sqrt(w @ cov_anual @ w)
    sharpe = (ret - TASA_LIBRE_RIESGO) / vol

    resultados[0, i] = ret
    resultados[1, i] = vol
    resultados[2, i] = sharpe
    pesos_mc[:, i]   = w

print(f'✅ {N_SIMULACIONES:,} portafolios simulados')

# Portafolio con máximo Sharpe Ratio
idx_max_sharpe = np.argmax(resultados[2])
idx_min_vol    = np.argmin(resultados[1])

print(f'\n🏆 Portafolio Máximo Sharpe:')
for j, t in enumerate(retornos.columns):
    print(f'   {t}: {pesos_mc[j, idx_max_sharpe]*100:.1f}%')
print(f'   → Retorno: {resultados[0, idx_max_sharpe]*100:.2f}%  |  Volatilidad: {resultados[1, idx_max_sharpe]*100:.2f}%  |  Sharpe: {resultados[2, idx_max_sharpe]:.3f}')

print(f'\n🛡️  Portafolio Mínima Volatilidad:')
for j, t in enumerate(retornos.columns):
    print(f'   {t}: {pesos_mc[j, idx_min_vol]*100:.1f}%')
print(f'   → Retorno: {resultados[0, idx_min_vol]*100:.2f}%  |  Volatilidad: {resultados[1, idx_min_vol]*100:.2f}%  |  Sharpe: {resultados[2, idx_min_vol]:.3f}')

In [ ]:
# ─── GRÁFICA: FRONTERA EFICIENTE ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

sc = ax.scatter(
    resultados[1] * 100,
    resultados[0] * 100,
    c=resultados[2], cmap='viridis',
    alpha=0.4, s=8
)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')

# Marcar portafolios óptimos
ax.scatter(
    resultados[1, idx_max_sharpe] * 100,
    resultados[0, idx_max_sharpe] * 100,
    color='red', s=200, zorder=5, marker='*',
    label=f'Máx. Sharpe ({resultados[2, idx_max_sharpe]:.2f})'
)
ax.scatter(
    resultados[1, idx_min_vol] * 100,
    resultados[0, idx_min_vol] * 100,
    color='blue', s=200, zorder=5, marker='D',
    label='Mín. Volatilidad'
)

# Activos individuales
for ticker in retornos.columns:
    ax.scatter(
        retornos[ticker].std() * np.sqrt(dias_trading) * 100,
        retornos[ticker].mean() * dias_trading * 100,
        s=120, zorder=6, marker='o'
    )
    ax.annotate(
        ticker,
        xy=(retornos[ticker].std() * np.sqrt(dias_trading) * 100,
            retornos[ticker].mean() * dias_trading * 100),
        xytext=(5, 5), textcoords='offset points', fontsize=10
    )

ax.set_xlabel('Volatilidad Anual (%)', fontsize=12)
ax.set_ylabel('Retorno Anual (%)', fontsize=12)
ax.set_title(f'Frontera Eficiente — Monte Carlo ({N_SIMULACIONES:,} portafolios)', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('img/03_frontera_eficiente.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfica guardada en img/')

## 6. Conclusiones

In [ ]:
# ─── RESUMEN EJECUTIVO ───────────────────────────────────────────────────────
mejor_sharpe_ticker = stats_df['Sharpe Ratio'].idxmax()
peor_sharpe_ticker  = stats_df['Sharpe Ratio'].idxmin()

print('=' * 60)
print('RESUMEN EJECUTIVO')
print('=' * 60)
print(f'\n📅 Período analizado: {FECHA_INICIO} → {FECHA_FIN}')
print(f'💰 Activos analizados: {len(retornos.columns)}')
print(f'\n🏆 Mejor Sharpe individual:  {mejor_sharpe_ticker} ({stats_df.loc[mejor_sharpe_ticker, "Sharpe Ratio"]:.2f})')
print(f'⚠️  Peor  Sharpe individual:  {peor_sharpe_ticker}  ({stats_df.loc[peor_sharpe_ticker, "Sharpe Ratio"]:.2f})')
print(f'\n🎯 Portafolio óptimo (Monte Carlo):')
print(f'   Sharpe:      {resultados[2, idx_max_sharpe]:.3f}')
print(f'   Retorno:     {resultados[0, idx_max_sharpe]*100:.2f}%')
print(f'   Volatilidad: {resultados[1, idx_max_sharpe]*100:.2f}%')
print(f'\n📌 Conclusión de diversificación:')
max_corr = corr.where(np.tril(np.ones(corr.shape), -1).astype(bool)).stack().idxmax()
min_corr = corr.where(np.tril(np.ones(corr.shape), -1).astype(bool)).stack().idxmin()
print(f'   Par más correlacionado:  {max_corr[0]}-{max_corr[1]} ({corr.loc[max_corr]:.2f})')
print(f'   Par menos correlacionado: {min_corr[0]}-{min_corr[1]} ({corr.loc[min_corr]:.2f})')